In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score


In [ ]:
print("SVM - COMPLAINT CLASSIFIER")
print("\nSTEP 1: Loading preprocessed dataset...")

data_path = "../data/preprocess_data/complaints_cleaned.csv"
results_dir = "../results/Dev"
model_dir = "../models/SVM"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

df = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Shape: {df.shape}")


In [ ]:
print("STEP 2: Preparing features and target...")

target_col = "Product"
primary_feature_col = "Issue"
secondary_feature_candidates = ["Sub-product", "Sub-issue"]
secondary_feature_col = next((c for c in secondary_feature_candidates if c in df.columns), None)

if secondary_feature_col is None:
    raise ValueError(f"Missing secondary feature column. Expected one of: {secondary_feature_candidates}")

required_cols = [primary_feature_col, secondary_feature_col, target_col]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df[primary_feature_col] = df[primary_feature_col].fillna("").astype(str)
df[secondary_feature_col] = df[secondary_feature_col].fillna("").astype(str)
df[target_col] = df[target_col].fillna("").astype(str)

X_text = (df[primary_feature_col] + " " + df[secondary_feature_col]).str.replace(r"\s+", " ", regex=True).str.strip()
y = df[target_col].str.replace(r"\s+", " ", regex=True).str.strip()

valid_mask = (X_text.str.len() > 0) & (y.str.len() > 0)
X_text = X_text[valid_mask]
y = y[valid_mask]

# Use a stratified sample for practical training time while keeping class balance.
max_rows_for_training = 100000
if len(X_text) > max_rows_for_training:
    X_text, _, y, _ = train_test_split(
        X_text,
        y,
        train_size=max_rows_for_training,
        stratify=y,
        random_state=42,
    )
    print(f"Using stratified sample: {max_rows_for_training:,} rows")

# Remove classes that are too small for a stratified split.
class_counts = y.value_counts()
low_support_classes = class_counts[class_counts < 2].index.tolist()
if low_support_classes:
    keep_mask = ~y.isin(low_support_classes)
    X_text = X_text[keep_mask]
    y = y[keep_mask]
    print(f"Dropped low-support classes: {low_support_classes}")

print(f"Features used: {[primary_feature_col, secondary_feature_col]}")
print(f"Target used: {target_col}")
print(f"Rows used: {len(X_text):,}")
print(f"Classes: {y.nunique()}")


In [ ]:
print("STEP 3: Train-test split...")

X_train, X_test, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Train size: {len(X_train):,}")
print(f"Test size: {len(X_test):,}")

vectorizer_kwargs = {
    "ngram_range": (1, 1),
    "min_df": 5,
    "max_df": 0.95,
    "max_features": 30000,
}

model = Pipeline([
    ("tfidf", TfidfVectorizer(**vectorizer_kwargs)),
    ("clf", LinearSVC(dual="auto", max_iter=2000)),
])

print("SVM pipeline ready.")


In [ ]:
print("STEP 4: Training model...")
model.fit(X_train, y_train)
print("✓ Training complete")

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

metrics = {
    "train_accuracy": accuracy_score(y_train, y_train_pred),
    "test_accuracy": accuracy_score(y_test, y_test_pred),
    "train_balanced_accuracy": balanced_accuracy_score(y_train, y_train_pred),
    "test_balanced_accuracy": balanced_accuracy_score(y_test, y_test_pred),
    "train_weighted_f1": f1_score(y_train, y_train_pred, average="weighted"),
    "test_weighted_f1": f1_score(y_test, y_test_pred, average="weighted"),
    "test_macro_f1": f1_score(y_test, y_test_pred, average="macro"),
}

print("\nEvaluation scores (%):")
for key, value in metrics.items():
    print(f"{key.replace('_', ' ').title()}: {value * 100:.2f}%")

report_text = classification_report(y_test, y_test_pred, zero_division=0)
print("\nClassification Report:")
print(report_text)


In [ ]:
print("STEP 5: Saving evaluation results and plots...")

metrics_df = pd.DataFrame([metrics])
metrics_df.insert(0, "model", ["LinearSVC"])
metrics_path = os.path.join(results_dir, "svm_evaluation_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)

report_path = os.path.join(results_dir, "svm_classification_report.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

predictions_df = pd.DataFrame({
    "actual": y_test.reset_index(drop=True),
    "predicted": pd.Series(y_test_pred),
})
predictions_path = os.path.join(results_dir, "svm_test_predictions.csv")
predictions_df.to_csv(predictions_path, index=False)

labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_test_pred, labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Normalized Confusion Matrix - Linear SVM")
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
plt.tight_layout()
cm_path = os.path.join(results_dir, "svm_confusion_matrix.png")
plt.savefig(cm_path, dpi=200, bbox_inches="tight")
plt.close(fig)

summary_path = os.path.join(results_dir, "svm_summary.txt")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("SVM Evaluation Summary\n")
    f.write("======================\n\n")
    for key, value in metrics.items():
        f.write(f"{key}: {value:.6f}\n")
    f.write("\nClassification Report:\n")
    f.write(report_text)
    f.write(f"\n\nConfusion matrix plot: {cm_path}\n")
    f.write(f"Metrics CSV: {metrics_path}\n")
    f.write(f"Predictions CSV: {predictions_path}\n")

print(f"Saved metrics to: {metrics_path}")
print(f"Saved report to: {report_path}")
print(f"Saved predictions to: {predictions_path}")
print(f"Saved confusion matrix to: {cm_path}")
print(f"Saved summary to: {summary_path}")


In [ ]:
print("STEP 6: Saving trained model...")

model_path = os.path.join(model_dir, "svm_pipeline.joblib")
joblib.dump(model, model_path)
print(f"Saved trained model to: {model_path}")

print("\nNotebook run complete.")
